# Aula 04: Lógica Proposicional — Conectivos Lógicos, Permissivos de Partida e Intertravamentos
## Engenharia de Controle e Automação — SCADA-Core Automática (Grupo 4)

## 1. Fundamentos Matemáticos: Conectivos Lógicos Proposicionais

Na matemática discreta, uma **proposição** é uma sentença declarativa $p$ que assume exatamente um valor-verdade em $\mathbb{B} = \{0, 1\}$ (ou $\{\text{Falso}, \text{Verdadeiro}\}$).

As operações entre proposições são regidas pelos conectivos fundamentais:

1. **Negação ($\neg p$ ou $\bar{p}$):** Inverte o valor-verdade do operando. Modela contatos normalmente fechados (NF / NC) e ausência de falha.
2. **Conjunção ($p \land q$):** Resulta em $1$ se e somente se ambos forem $1$. Em automação, modela circuitos e condições em **série** (todos os requisitos devem ser satisfeitos simultaneamente).
3. **Disjunção ($p \lor q$):** Resulta em $1$ se pelo menos um dos operandos for $1$. Modela circuitos em **paralelo** ou múltiplas causas de desarme/falha (*Trip*).
4. **Disjunção Exclusiva ($p \oplus q$):** Resulta em $1$ se os operandos tiverem valores distintos ($p \neq q$). Modela chaves seletoras e modos mutuamente exclusivos (ex: Manual vs Automático).
5. **Condicional / Implicação ($p \rightarrow q$):** É falsa exclusivamente quando o antecedente é verdadeiro e o consequente é falso ($p \land \neg q$). É logicamente equivalente a $\neg p \lor q$. Modela regras de controle: *"SE ocorrer sobrecarga, ENTÃO desarme o disjuntor"*.
6. **Bicondicional ($p \leftrightarrow q$):** Resulta em $1$ se ambos tiverem o mesmo valor lógico ($p = q$). Equivalente a $(p \rightarrow q) \land (q \rightarrow p)$.

In [1]:
import itertools
from typing import Callable, List, Dict, Any

# =============================================================================
# 1. OPERADORES PRIMITIVOS DA LÓGICA PROPOSICIONAL (MATEMÁTICA DISCRETA)
# =============================================================================

def NOT(p: bool) -> bool:
    """Negação Proposicional (¬p)"""
    return not p

def AND(p: bool, q: bool) -> bool:
    """Conjunção Proposicional (p ∧ q)"""
    return p and q

def OR(p: bool, q: bool) -> bool:
    """Disjunção Proposicional (p ∨ q)"""
    return p or q

def XOR(p: bool, q: bool) -> bool:
    """Disjunção Exclusiva (p ⊕ q)"""
    return p ^ q

def IMPLIES(p: bool, q: bool) -> bool:
    """Condicional / Implicação Proposicional (p → q ≡ ¬p ∨ q)"""
    return (not p) or q

def IFF(p: bool, q: bool) -> bool:
    """Bicondicional / Equivalência Proposicional (p ↔ q ≡ (p → q) ∧ (q → p))"""
    return p == q

print("[OK] Conectivos logicos fundamentais carregados com sucesso.")

[OK] Conectivos logicos fundamentais carregados com sucesso.


### 1.1 Tabelas-Verdade dos Operadores Lógicos

A seguir, geramos as tabelas-verdade formais para todos os operadores binários e unários em notação booleana ($1$ para Verdadeiro e $0$ para Falso).

In [8]:
def exibir_tabela_verdade_binaria(nome_operador: str, fn: Callable[[bool, bool], bool]):
    """Gera e exibe a tabela-verdade formal para um operador lógico binário."""
    print(f"=== Tabela-Verdade: {nome_operador} ===")
    print(f"|  A  |  B  | Resultado (Binario) | Booleano |")
    print(f"|:---:|:---:|:-------------------:|:--------:|")
    for a, b in [(1, 1), (1, 0), (0, 1), (0, 0)]:
        res = fn(bool(a), bool(b))
        res_bin = 1 if res else 0
        print(f"|  {a}  |  {b}  |          {res_bin}          |  {str(res):<5}   |")
    print()

# Tabela da Negação (Unária)
print("=== Tabela-Verdade: Negacao (~A) ===")
print("|  A  |  ~A (Binario)  | Booleano |")
print("|:---:|:--------------:|:--------:|")
for a in [1, 0]:
    res = NOT(bool(a))
    print(f"|  {a}  |       {1 if res else 0}        |  {str(res):<5}   |")
print()

# Tabelas dos Operadores Binários
exibir_tabela_verdade_binaria("Conjuncao (A /\\ B)", AND)
exibir_tabela_verdade_binaria("Disjuncao (A \\/ B)", OR)
exibir_tabela_verdade_binaria("Disjuncao Exclusiva (A XOR B)", XOR)
exibir_tabela_verdade_binaria("Condicional / Implicacao (A -> B)", IMPLIES)
exibir_tabela_verdade_binaria("Bicondicional (A <-> B)", IFF)

=== Tabela-Verdade: Negacao (~A) ===
|  A  |  ~A (Binario)  | Booleano |
|:---:|:--------------:|:--------:|
|  1  |       0        |  False   |
|  0  |       1        |  True    |

=== Tabela-Verdade: Conjuncao (A /\ B) ===
|  A  |  B  | Resultado (Binario) | Booleano |
|:---:|:---:|:-------------------:|:--------:|
|  1  |  1  |          1          |  True    |
|  1  |  0  |          0          |  False   |
|  0  |  1  |          0          |  False   |
|  0  |  0  |          0          |  False   |

=== Tabela-Verdade: Disjuncao (A \/ B) ===
|  A  |  B  | Resultado (Binario) | Booleano |
|:---:|:---:|:-------------------:|:--------:|
|  1  |  1  |          1          |  True    |
|  1  |  0  |          1          |  True    |
|  0  |  1  |          1          |  True    |
|  0  |  0  |          0          |  False   |

=== Tabela-Verdade: Disjuncao Exclusiva (A XOR B) ===
|  A  |  B  | Resultado (Binario) | Booleano |
|:---:|:---:|:-------------------:|:--------:|
|  1  |  1  |     

## 2. Leis de De Morgan e Álgebra Booleana Aplicada

As Leis de De Morgan são fundamentais na engenharia para a transformação e simplificação de circuitos e lógicas de intertravamento:

1. **Primeira Lei de De Morgan (Negação da Conjunção):**
   $$\neg (A \land B) \equiv \neg A \lor \neg B$$
   *Significado em Automação:* Dizer que "um conjunto de condições simultâneas de segurança falhou" equivale a dizer que "pelo menos uma das condições falhou".

2. **Segunda Lei de De Morgan (Negação da Disjunção):**
   $$\neg (A \lor B) \equiv \neg A \land \neg B$$
   *Significado em Automação:* Dizer que "nenhuma falha ocorreu" equivale a dizer que "a primeira falha não ocorreu E a segunda falha não ocorreu".

Vamos provar computacionalmente essas equivalências avaliando se a bicondicional entre ambos os lados é uma **tautologia** (sempre verdadeira para qualquer combinação).

In [11]:
def verificar_tautologia_2vars(expr1: Callable[[bool, bool], bool], expr2: Callable[[bool, bool], bool], nome: str):
    """Testa se expr1 <-> expr2 é uma Tautologia em todo o espaço de estados 2^2."""
    todas_iguais = True
    print(f"Testando Equivalencia Logica: {nome}")
    print("|  A  |  B  | Lado Esquerdo | Lado Direito | Equivalente? |")
    print("|:---:|:---:|:-------------:|:------------:|:------------:|")
    for a, b in itertools.product([True, False], repeat=2):
        lhs = expr1(a, b)
        rhs = expr2(a, b)
        eq = IFF(lhs, rhs)
        if not eq:
            todas_iguais = False
        print(f"|  {int(a)}  |  {int(b)}  |       {int(lhs)}       |      {int(rhs)}       |    {str(eq):<5}     |")

    if todas_iguais:
        print(f"[PROVADO]: '{nome}' e uma TAUTOLOGIA (Equivalencia Formal Valida!)\n")
    else:
        print(f"[ERRO]: A equivalencia falhou!\n")

# 1. 1ª Lei de De Morgan: ¬(A ∧ B) ≡ ¬A ∨ ¬B
verificar_tautologia_2vars(
    lambda a, b: NOT(AND(a, b)),
    lambda a, b: OR(NOT(a), NOT(b)),
    "~(A /\\ B) = ~A \\/ ~B (1a Lei de De Morgan)"
)

# 2. 2ª Lei de De Morgan: ¬(A ∨ B) ≡ ¬A ∧ ¬B
verificar_tautologia_2vars(
    lambda a, b: NOT(OR(a, b)),
    lambda a, b: AND(NOT(a), NOT(b)),
    "~(A \\/ B) = ~A /\\ ~B (2a Lei de De Morgan)"
)

# 3. Equivalência da Implicação: (A → B) ≡ (¬A ∨ B)
verificar_tautologia_2vars(
    lambda a, b: IMPLIES(a, b),
    lambda a, b: OR(NOT(a), b),
    "(A -> B) = (~A \\/ B) (Definicao da Implicacao)"
)

Testando Equivalencia Logica: ~(A /\ B) = ~A \/ ~B (1a Lei de De Morgan)
|  A  |  B  | Lado Esquerdo | Lado Direito | Equivalente? |
|:---:|:---:|:-------------:|:------------:|:------------:|
|  1  |  1  |       0       |      0       |    True      |
|  1  |  0  |       1       |      1       |    True      |
|  0  |  1  |       1       |      1       |    True      |
|  0  |  0  |       1       |      1       |    True      |
[PROVADO]: '~(A /\ B) = ~A \/ ~B (1a Lei de De Morgan)' e uma TAUTOLOGIA (Equivalencia Formal Valida!)

Testando Equivalencia Logica: ~(A \/ B) = ~A /\ ~B (2a Lei de De Morgan)
|  A  |  B  | Lado Esquerdo | Lado Direito | Equivalente? |
|:---:|:---:|:-------------:|:------------:|:------------:|
|  1  |  1  |       0       |      0       |    True      |
|  1  |  0  |       0       |      0       |    True      |
|  0  |  1  |       0       |      0       |    True      |
|  0  |  0  |       1       |      1       |    True      |
[PROVADO]: '~(A \/ B) = ~A /\ 

## 3. Aplicação em Engenharia: Permissivos de Partida e Intertravamentos da Planta

### 3.1 Distinção Fundamental de Engenharia de Automação:
- **Permissivo de Partida (*Start Permissive*):** Condições booleanas prévias que devem ser atendidas para que o operador ou sistema SCADA possa dar o comando de inicialização a um equipamento (ex.: funil com nível, esteira ligada, pressão pneumática correta).
- **Intertravamento de Operação / Trip (*Run Interlock / Emergency Trip*):** Condição dinâmica contínua monitorada a cada ciclo de varredura (*scan cycle*) do CLP. Se violada durante a marcha, causa a desenergização imediata e segura dos atuadores.

---

### 3.2 Mapeamento das Equações Lógicas da Planta de Grãos

Consolidando as especificações dos documentos `00 - Descritivo Do Processo.md`, `01 - Variáveis do Processo.md` e `04 - Logica Proposicional Conectivos e Permissivos.md`:

#### A. Permissão Geral de Operação ($c_{\text{PERM}}$)
$$c_{\text{PERM}} \equiv \neg p_{\text{EMERG}} \land \neg p_{\text{JI201}} \land \neg p_{\text{PAL601}} \land \neg p_{\text{NC703}} \land p_{\text{KSA401}}$$

#### B. Permissivo do Alimentador Vibratório ($P_{\text{ALIM}}$)
$$P_{\text{ALIM}} \equiv c_{\text{PERM}} \land p_{\text{MOV201}} \land \neg p_{\text{NB101}}$$
Expandido:
$$P_{\text{ALIM}} \equiv (\neg p_{\text{EMERG}} \land \neg p_{\text{JI201}} \land \neg p_{\text{PAL601}} \land \neg p_{\text{NC703}} \land p_{\text{KSA401}}) \land p_{\text{MOV201}} \land \neg p_{\text{NB101}}$$

#### C. Classificação Lógica dos Grãos por Visão Computacional
- **Categoria A (Aprovado Integralmente):**
  $$p_{\text{A}} \equiv p_{\text{CV101}} \land p_{\text{CV103}} \land p_{\text{CV105}} \land \neg p_{\text{CV107}} \land \neg p_{\text{CV108}} \land \neg p_{\text{CV109}}$$
- **Categoria C (Rejeitado - Defeito Grave ou Fora de Padrão):**
  $$p_{\text{C}} \equiv p_{\text{CV107}} \lor p_{\text{CV108}} \lor p_{\text{CV109}} \lor (\neg p_{\text{CV101}} \land \neg p_{\text{CV102}}) \lor (\neg p_{\text{CV103}} \land \neg p_{\text{CV104}}) \lor (\neg p_{\text{CV105}} \land \neg p_{\text{CV106}})$$
- **Categoria B (Secundário - Partição por Exclusão):**
  $$p_{\text{B}} \equiv \neg p_{\text{A}} \land \neg p_{\text{C}}$$

#### D. Ejeção Pneumática e Diagnóstico de Falha
- **Comando de Disparo da Válvula Ejetora ($c_{\text{FY603}}$):**
  $$c_{\text{FY603}} \equiv p_{\text{C}} \land p_{\text{POS603}} \land \neg p_{\text{PAL601}}$$
- **Diagnóstico de Falha do Atuador ($p_{\text{FALHA-EJETOR}}$):**
  $$p_{\text{FALHA-EJETOR}} \equiv c_{\text{FY603}} \land \neg p_{\text{ZSH601}}$$

In [4]:
# =============================================================================
# 2. IMPLEMENTAÇÃO DAS REGRAS LÓGICAS DA PLANTA INDUSTRIAL
# =============================================================================

def c_PERM(v: Dict[str, bool]) -> bool:
    """
    Permissão Geral de Operação consolidada (c_PERM).
    Exige ausência de emergência, motor sem sobrecarga, ar comprimido OK,
    reservatório de rejeito sem transbordo e câmera pronta.
    """
    return (
        NOT(v['p_EMERG'])
        and NOT(v['p_JI201'])
        and NOT(v['p_PAL601'])
        and NOT(v['p_NC703'])
        and v['p_KSA401']
    )

def P_ALIM(v: Dict[str, bool]) -> bool:
    """
    Permissivo de Partida do Alimentador Vibratório.
    Exige Permissão Geral OK, Esteira em movimento e Funil com nível suficiente.
    """
    return c_PERM(v) and v['p_MOV201'] and NOT(v['p_NB101'])

def p_A(v: Dict[str, bool]) -> bool:
    """Classificação Categoria A (Aprovado): todos parâmetros ideais e sem defeitos."""
    return (
        v['p_CV101'] and v['p_CV103'] and v['p_CV105']
        and NOT(v['p_CV107']) and NOT(v['p_CV108']) and NOT(v['p_CV109'])
    )

def p_C(v: Dict[str, bool]) -> bool:
    """Classificação Categoria C (Rejeitado): defeito grave ou fora da tolerância."""
    defeito = v['p_CV107'] or v['p_CV108'] or v['p_CV109']
    cor_invalida = NOT(v['p_CV101']) and NOT(v['p_CV102'])
    tam_invalido = NOT(v['p_CV103']) and NOT(v['p_CV104'])
    for_invalido = NOT(v['p_CV105']) and NOT(v['p_CV106'])
    return defeito or cor_invalida or tam_invalido or for_invalido

def p_B(v: Dict[str, bool]) -> bool:
    """Classificação Categoria B (Secundário): nem A, nem C."""
    return NOT(p_A(v)) and NOT(p_C(v))

def c_FY603(v: Dict[str, bool]) -> bool:
    """Comando de Disparo da Válvula Ejetora Pneumática."""
    return p_C(v) and v['p_POS603'] and NOT(v['p_PAL601'])

def p_FALHA_EJETOR(v: Dict[str, bool]) -> bool:
    """Diagnóstico de Falha do Ejetor: comando acionado sem confirmação física."""
    return c_FY603(v) and NOT(v['p_ZSH601'])

print("[OK] Equacoes logicas do processo industrial compiladas com sucesso!")

[OK] Equacoes logicas do processo industrial compiladas com sucesso!


## 4. Demonstração Analítica e Computacional das Condições de Bloqueio e Trip

### 4.1 Relação de Bloqueio do Atuador Ejetor (De Morgan)
A condição de **bloqueio ou inibição** da válvula solenoide de ejeção é a negação do seu sinal de comando:
$$\text{Bloqueio}_{\text{FY603}} \equiv \neg c_{\text{FY603}} = \neg (p_{\text{C}} \land p_{\text{POS603}} \land \neg p_{\text{PAL601}})$$

Aplicando a 1ª Lei de De Morgan e eliminando a dupla negação:
$$\text{Bloqueio}_{\text{FY603}} \equiv \neg p_{\text{C}} \lor \neg p_{\text{POS603}} \lor p_{\text{PAL601}}$$

*Interpretação:* A ejeção é bloqueada se o grão **não** for rejeitado ($p_{\text{C}}=0$), **ou** se o grão **não** estiver na posição do bocal ($p_{\text{POS603}}=0$), **ou** se houver **falha de pressão** pneumática ($p_{\text{PAL601}}=1$).

---

### 4.2 Condição Geral de Parada de Emergência / Trip do Processo
O desarme geral da planta (\text{Trip}_{\text{GERAL}}) ocorre quando a permissão geral é revogada:
$$\text{Trip}_{\text{GERAL}} \equiv \neg c_{\text{PERM}} = \neg (\neg p_{\text{EMERG}} \land \neg p_{\text{JI201}} \land \neg p_{\text{PAL601}} \land \neg p_{\text{NC703}} \land p_{\text{KSA401}})$$

Aplicando De Morgan:
$$\text{Trip}_{\text{GERAL}} \equiv p_{\text{EMERG}} \lor p_{\text{JI201}} \lor p_{\text{PAL601}} \lor p_{\text{NC703}} \lor \neg p_{\text{KSA401}}$$

*Interpretação:* Qualquer falha individual dispara imediatamente o intertravamento geral da planta.

In [12]:
# =============================================================================
# 3. PROVA COMPUTACIONAL DAS FORMAS NEGADAS VIA DE MORGAN
# =============================================================================

# Prova 1: Bloqueio do Ejetor
prova1_ok = True
for pC, pPOS, pPAL in itertools.product([False, True], repeat=3):
    cmd_direto = pC and pPOS and (not pPAL)
    bloqueio_direto = not cmd_direto
    bloqueio_demorgan = (not pC) or (not pPOS) or pPAL
    if bloqueio_direto != bloqueio_demorgan:
        prova1_ok = False

print("1. Prova do Bloqueio do Ejetor ~c_FY603 = (~p_C \\/ ~p_POS603 \\/ p_PAL601):")
print(f"   Resultado: {'[PROVADO]: Tautologia Rigorosamente Valida' if prova1_ok else '[FALHA]'}\n")

# Prova 2: Trip Geral da Planta
prova2_ok = True
for pEMERG, pJI, pPAL, pNC, pKSA in itertools.product([False, True], repeat=5):
    perm = (not pEMERG) and (not pJI) and (not pPAL) and (not pNC) and pKSA
    trip_direto = not perm
    trip_demorgan = pEMERG or pJI or pPAL or pNC or (not pKSA)
    if trip_direto != trip_demorgan:
        prova2_ok = False

print("2. Prova do Trip Geral ~c_PERM = (p_EMERG \\/ p_JI201 \\/ p_PAL601 \\/ p_NC703 \\/ ~p_KSA401):")
print(f"   Resultado: {'[PROVADO]: Tautologia Rigorosamente Valida' if prova2_ok else '[FALHA]'}")

1. Prova do Bloqueio do Ejetor ~c_FY603 = (~p_C \/ ~p_POS603 \/ p_PAL601):
   Resultado: [PROVADO]: Tautologia Rigorosamente Valida

2. Prova do Trip Geral ~c_PERM = (p_EMERG \/ p_JI201 \/ p_PAL601 \/ p_NC703 \/ ~p_KSA401):
   Resultado: [PROVADO]: Tautologia Rigorosamente Valida


## 5. Tabela-Verdade Exaustiva da Permissão Geral e Permissivo do Alimentador

Avaliamos agora todas as $2^5 = 32$ combinações de entrada da Permissão Geral ($c_{\text{PERM}}$) e as $2^7 = 128$ combinações do Permissivo de Partida do Alimentador ($P_{\text{ALIM}}$).

In [6]:
# =============================================================================
# 4. TABELA-VERDADE EXAUSTIVA DE c_PERM (32 COMBINAÇÕES)
# =============================================================================

entradas_perm = ['p_EMERG', 'p_JI201', 'p_PAL601', 'p_NC703', 'p_KSA401']
linhas_perm = []

for combo in itertools.product([0, 1], repeat=len(entradas_perm)):
    estado = {var: bool(val) for var, val in zip(entradas_perm, combo)}
    res = c_PERM(estado)
    linhas_perm.append({**estado, 'c_PERM': res})

total_comb = len(linhas_perm)
comb_ativas = sum(1 for row in linhas_perm if row['c_PERM'])

print(f"=== ANALISE DE SEGURANCA DE c_PERM ===")
print(f"Total de combinacoes de falha/operacao: {total_comb}")
print(f"Combinacoes seguras que liberam a planta (c_PERM = 1): {comb_ativas} (apenas {comb_ativas/total_comb*100:.1f}% do espaco de estados!)")
print(f"Combinacoes que bloqueiam a planta (c_PERM = 0): {total_comb - comb_ativas}\n")

print("Amostra da Tabela-Verdade (Primeiras 8 linhas + Linha Ativa):")
print("| EMERG | JI201 | PAL601 | NC703 | KSA401 | c_PERM (Saida) |")
print("|:-----:|:-----:|:------:|:-----:|:------:|:--------------:|")
for row in linhas_perm[:8] + [r for r in linhas_perm if r['c_PERM']]:
    print(f"|   {int(row['p_EMERG'])}   |   {int(row['p_JI201'])}   |   {int(row['p_PAL601'])}    |   {int(row['p_NC703'])}   |   {int(row['p_KSA401'])}    |       {int(row['c_PERM'])}        |")

=== ANALISE DE SEGURANCA DE c_PERM ===
Total de combinacoes de falha/operacao: 32
Combinacoes seguras que liberam a planta (c_PERM = 1): 1 (apenas 3.1% do espaco de estados!)
Combinacoes que bloqueiam a planta (c_PERM = 0): 31

Amostra da Tabela-Verdade (Primeiras 8 linhas + Linha Ativa):
| EMERG | JI201 | PAL601 | NC703 | KSA401 | c_PERM (Saida) |
|:-----:|:-----:|:------:|:-----:|:------:|:--------------:|
|   0   |   0   |   0    |   0   |   0    |       0        |
|   0   |   0   |   0    |   0   |   1    |       1        |
|   0   |   0   |   0    |   1   |   0    |       0        |
|   0   |   0   |   0    |   1   |   1    |       0        |
|   0   |   0   |   1    |   0   |   0    |       0        |
|   0   |   0   |   1    |   0   |   1    |       0        |
|   0   |   0   |   1    |   1   |   0    |       0        |
|   0   |   0   |   1    |   1   |   1    |       0        |
|   0   |   0   |   0    |   0   |   1    |       1        |


## 6. Simulação de Cenários Industriais Realistas

Para consolidar a prática profissional, testamos o comportamento do sistema de controle sob 10 cenários operacionais reais da indústria.

In [7]:
# =============================================================================
# 5. SIMULAÇÃO DE CENÁRIOS OPERACIONAIS (MATRIZ DE CAUSA E EFEITO)
# =============================================================================

cenarios = [
    {
        "nome": "01. Operacao Nominal (Grao Perfeito A)",
        "desc": "Toda a planta normal, esteira rodando, funil cheio, grao sem defeitos.",
        "tags": {
            'p_EMERG': False, 'p_JI201': False, 'p_PAL601': False, 'p_NC703': False, 'p_KSA401': True,
            'p_MOV201': True, 'p_NB101': False, 'p_POS603': True, 'p_ZSH601': False,
            'p_CV101': True, 'p_CV102': False, 'p_CV103': True, 'p_CV104': False,
            'p_CV105': True, 'p_CV106': False, 'p_CV107': False, 'p_CV108': False, 'p_CV109': False
        }
    },
    {
        "nome": "02. Emergencia Acionada (XA-901)",
        "desc": "Operador pressionou o botao de emergencia.",
        "tags": {
            'p_EMERG': True, 'p_JI201': False, 'p_PAL601': False, 'p_NC703': False, 'p_KSA401': True,
            'p_MOV201': True, 'p_NB101': False, 'p_POS603': False, 'p_ZSH601': False,
            'p_CV101': True, 'p_CV102': False, 'p_CV103': True, 'p_CV104': False,
            'p_CV105': True, 'p_CV106': False, 'p_CV107': False, 'p_CV108': False, 'p_CV109': False
        }
    },
    {
        "nome": "03. Sobrecarga no Motor da Esteira (JI-201)",
        "desc": "Rele termico acusa corrente excessiva no acionamento da esteira.",
        "tags": {
            'p_EMERG': False, 'p_JI201': True, 'p_PAL601': False, 'p_NC703': False, 'p_KSA401': True,
            'p_MOV201': False, 'p_NB101': False, 'p_POS603': False, 'p_ZSH601': False,
            'p_CV101': True, 'p_CV102': False, 'p_CV103': True, 'p_CV104': False,
            'p_CV105': True, 'p_CV106': False, 'p_CV107': False, 'p_CV108': False, 'p_CV109': False
        }
    },
    {
        "nome": "04. Queda de Pressao Pneumatica (PAL-601)",
        "desc": "Pressostato indica pressao na linha pneumatica inferior a 6 bar.",
        "tags": {
            'p_EMERG': False, 'p_JI201': False, 'p_PAL601': True, 'p_NC703': False, 'p_KSA401': True,
            'p_MOV201': True, 'p_NB101': False, 'p_POS603': True, 'p_ZSH601': False,
            'p_CV101': False, 'p_CV102': False, 'p_CV103': False, 'p_CV104': False,
            'p_CV105': False, 'p_CV106': False, 'p_CV107': True, 'p_CV108': False, 'p_CV109': False
        }
    },
    {
        "nome": "05. Reservatorio de Rejeito Cheio 100% (NC-703)",
        "desc": "Transmissor de nivel acusa risco iminente de transbordo no silo de descarte.",
        "tags": {
            'p_EMERG': False, 'p_JI201': False, 'p_PAL601': False, 'p_NC703': True, 'p_KSA401': True,
            'p_MOV201': True, 'p_NB101': False, 'p_POS603': False, 'p_ZSH601': False,
            'p_CV101': True, 'p_CV102': False, 'p_CV103': True, 'p_CV104': False,
            'p_CV105': True, 'p_CV106': False, 'p_CV107': False, 'p_CV108': False, 'p_CV109': False
        }
    },
    {
        "nome": "06. Funil de Graos Vazio (p_NB101)",
        "desc": "Nivel baixo no funil de recepcao impede partida a seco do alimentador vibratorio.",
        "tags": {
            'p_EMERG': False, 'p_JI201': False, 'p_PAL601': False, 'p_NC703': False, 'p_KSA401': True,
            'p_MOV201': True, 'p_NB101': True, 'p_POS603': False, 'p_ZSH601': False,
            'p_CV101': True, 'p_CV102': False, 'p_CV103': True, 'p_CV104': False,
            'p_CV105': True, 'p_CV106': False, 'p_CV107': False, 'p_CV108': False, 'p_CV109': False
        }
    },
    {
        "nome": "07. Grao Defeituoso (Cat C) Ejecao Bem-Sucedida",
        "desc": "Grao com praga e dano atinge o bocal. Ejetor acionado com ar OK e avanco mecanico confirmado.",
        "tags": {
            'p_EMERG': False, 'p_JI201': False, 'p_PAL601': False, 'p_NC703': False, 'p_KSA401': True,
            'p_MOV201': True, 'p_NB101': False, 'p_POS603': True, 'p_ZSH601': True,
            'p_CV101': False, 'p_CV102': False, 'p_CV103': True, 'p_CV104': False,
            'p_CV105': True, 'p_CV106': False, 'p_CV107': True, 'p_CV108': True, 'p_CV109': False
        }
    },
    {
        "nome": "08. Falha Mecanica do Ejetor (p_FALHA_EJETOR)",
        "desc": "Valvula solenoide acionada para grao Cat C, mas sensor ZSH-601 nao confirma curso mecanico.",
        "tags": {
            'p_EMERG': False, 'p_JI201': False, 'p_PAL601': False, 'p_NC703': False, 'p_KSA401': True,
            'p_MOV201': True, 'p_NB101': False, 'p_POS603': True, 'p_ZSH601': False,
            'p_CV101': False, 'p_CV102': False, 'p_CV103': True, 'p_CV104': False,
            'p_CV105': True, 'p_CV106': False, 'p_CV107': True, 'p_CV108': False, 'p_CV109': False
        }
    },
    {
        "nome": "09. Grao Categoria B (Qualidade Comercial Secundaria)",
        "desc": "Grao com cor secundaria (CV-102), sem pragas/danos. Nao ejeta, segue na esteira.",
        "tags": {
            'p_EMERG': False, 'p_JI201': False, 'p_PAL601': False, 'p_NC703': False, 'p_KSA401': True,
            'p_MOV201': True, 'p_NB101': False, 'p_POS603': True, 'p_ZSH601': False,
            'p_CV101': False, 'p_CV102': True, 'p_CV103': True, 'p_CV104': False,
            'p_CV105': True, 'p_CV106': False, 'p_CV107': False, 'p_CV108': False, 'p_CV109': False
        }
    },
    {
        "nome": "10. Inibicao de Ejecao por Queda de Pressao",
        "desc": "Grao Cat C detectado na posicao, porem pressao baixa inibe acionamento do bocal.",
        "tags": {
            'p_EMERG': False, 'p_JI201': False, 'p_PAL601': True, 'p_NC703': False, 'p_KSA401': True,
            'p_MOV201': True, 'p_NB101': False, 'p_POS603': True, 'p_ZSH601': False,
            'p_CV101': False, 'p_CV102': False, 'p_CV103': False, 'p_CV104': False,
            'p_CV105': False, 'p_CV106': False, 'p_CV107': True, 'p_CV108': False, 'p_CV109': False
        }
    }
]

print("=" * 100)
print(f"{'SIMULACAO DOS CENARIOS OPERACIONAIS DA PLANTA SCADA':^100}")
print("=" * 100)

for c in cenarios:
    t = c["tags"]
    perm = c_PERM(t)
    alim = P_ALIM(t)
    cat_a = p_A(t)
    cat_b = p_B(t)
    cat_c = p_C(t)
    ejetor = c_FY603(t)
    falha_ej = p_FALHA_EJETOR(t)

    cat_str = "A (Aprovado)" if cat_a else ("B (Secundario)" if cat_b else "C (Rejeitado)")

    print(f"\n>> CENARIO: {c['nome']}")
    print(f"   Descricao: {c['desc']}")
    print(f"   ----------------------------------------------------------------------------------------")
    print(f"   [c_PERM: {'LIBERADA' if perm else 'BLOQUEADA':<9}]  "
          f"[P_ALIM: {'PERMITIDO' if alim else 'IMPEDIDO':<9}]  "
          f"[Classificacao: {cat_str:<15}]  "
          f"[Ejetor FY-603: {'DISPARAR' if ejetor else 'DESLIGADO':<9}]  "
          f"[Alarme Ejetor: {'FALHA!' if falha_ej else 'OK':<6}]")

                        SIMULACAO DOS CENARIOS OPERACIONAIS DA PLANTA SCADA                         

>> CENARIO: 01. Operacao Nominal (Grao Perfeito A)
   Descricao: Toda a planta normal, esteira rodando, funil cheio, grao sem defeitos.
   ----------------------------------------------------------------------------------------
   [c_PERM: LIBERADA ]  [P_ALIM: PERMITIDO]  [Classificacao: A (Aprovado)   ]  [Ejetor FY-603: DESLIGADO]  [Alarme Ejetor: OK    ]

>> CENARIO: 02. Emergencia Acionada (XA-901)
   Descricao: Operador pressionou o botao de emergencia.
   ----------------------------------------------------------------------------------------
   [c_PERM: BLOQUEADA]  [P_ALIM: IMPEDIDO ]  [Classificacao: A (Aprovado)   ]  [Ejetor FY-603: DESLIGADO]  [Alarme Ejetor: OK    ]

>> CENARIO: 03. Sobrecarga no Motor da Esteira (JI-201)
   Descricao: Rele termico acusa corrente excessiva no acionamento da esteira.
   --------------------------------------------------------------------------